# Chapter 9: Sequence Models — Processing Language in Order

Regular neural networks see all inputs at once. But language has **order**:

- "Dog bites man" ≠ "Man bites dog" (same words, different meaning)
- "I am not happy" — the "not" changes everything

Sequence models process inputs **one at a time, remembering what came before**.

```
Regular network:   [all words] → prediction
Sequence model:    word1 → word2 → word3 → ... → prediction
                   (carries memory forward at each step)
```

---
## Recurrent Neural Network (RNN): The Basic Idea

An RNN has a **hidden state** that acts as memory. At each step:

1. Take the current input AND the previous hidden state
2. Produce a new hidden state
3. Pass that hidden state to the next step

```
         h0        h1        h2        h3
          │         │         │         │
    ┌───┐ ↓   ┌───┐ ↓   ┌───┐ ↓   ┌───┐ ↓
x0→ │RNN│──→  │RNN│──→  │RNN│──→  │RNN│──→ output
    └───┘     └───┘     └───┘     └───┘
    "the"     "cat"     "sat"     "down"
```

The SAME RNN cell is reused at every step (shared weights).

In [ ]:
import numpy as np

np.random.seed(42)

def tanh(x):
    return np.tanh(x)

# Simplest RNN cell
def rnn_cell(x, h_prev, W_xh, W_hh, b_h):
    """One step of an RNN.
    x:      current input
    h_prev: previous hidden state (memory)
    Returns: new hidden state
    """
    h_new = tanh(x @ W_xh + h_prev @ W_hh + b_h)
    return h_new

# Dimensions
input_size = 3    # each word is a 3-number vector
hidden_size = 4   # memory has 4 numbers

# Weights (shared across all time steps)
W_xh = np.random.randn(input_size, hidden_size) * 0.3
W_hh = np.random.randn(hidden_size, hidden_size) * 0.3
b_h = np.zeros((1, hidden_size))

# Fake word vectors for "the cat sat"
words = {
    "the": np.array([[0.1, 0.2, 0.0]]),
    "cat": np.array([[0.8, 0.1, 0.9]]),
    "sat": np.array([[0.3, 0.7, 0.2]]),
}

print("RNN Processing: 'the cat sat'\n")
print(f"Hidden state size: {hidden_size} numbers")
print(f"Same weights used at every step\n")

h = np.zeros((1, hidden_size))  # initial hidden state = zeros

for word, x in words.items():
    h_prev = h.copy()
    h = rnn_cell(x, h, W_xh, W_hh, b_h)
    print(f"  Step '{word}':")
    print(f"    Input:          {x.flatten().round(2)}")
    print(f"    Prev hidden:    {h_prev.flatten().round(3)}")
    print(f"    New hidden:     {h.flatten().round(3)}")
    print(f"    (memory now encodes everything seen so far)")
    print()

print("Final hidden state encodes the ENTIRE sequence 'the cat sat'.")
print("This can be fed to a dense layer for classification or prediction.")

---
## The Problem: Vanishing Memory

RNNs have a critical weakness: they **forget early inputs** in long sequences.

```
"I grew up in France. I went to school there. I learned to cook.
 Years later, I moved to Spain. ... I speak fluent ____"
```

By the time we reach the blank, "France" (50 words ago) has faded from the hidden state.

This is the **vanishing gradient problem** applied to sequences — gradients shrink as they travel back through many time steps.

In [ ]:
# Demonstrate vanishing memory
print("How RNN Memory Fades Over Time:\n")

h = np.zeros((1, hidden_size))
# Inject a strong signal at step 0
strong_input = np.array([[1.0, 1.0, 1.0]])
h = rnn_cell(strong_input, h, W_xh, W_hh, b_h)
initial_strength = np.linalg.norm(h)

print(f"Step  0: input=[1,1,1] (strong signal)  memory strength: {initial_strength:.3f}")

# Feed zeros for many steps (no new info, just decay)
zero_input = np.array([[0.0, 0.0, 0.0]])
for step in range(1, 21):
    h = rnn_cell(zero_input, h, W_xh, W_hh, b_h)
    strength = np.linalg.norm(h)
    bar = "█" * int(strength / initial_strength * 30)
    print(f"Step {step:>2}: input=[0,0,0] (nothing new)    memory strength: {strength:.3f}  {bar}")

print(f"\nThe initial signal ({initial_strength:.3f}) has faded to {strength:.3f}.")
print("This is why basic RNNs can't handle long sequences.")

---
## LSTM: Long Short-Term Memory

LSTM solves the vanishing memory problem with **gates** that control information flow.

Think of it like a conveyor belt: information flows forward unchanged unless a gate actively modifies it.

Three gates:
- **Forget gate**: what to erase from memory ("forget that the subject was singular")
- **Input gate**: what new info to store ("remember this new subject")
- **Output gate**: what to output from memory ("use the subject for the verb")

```
         ┌─────── Cell State (long-term memory) ──────────┐
         │    ×forget    +input                            │
         │      ↑          ↑                               │
  h_prev─┤  [forget]  [input]  [output]                   ├─ h_new
         │   gate       gate     gate                      │
  x_t ───┘                        │                        │
                                  ↓                        │
                          tanh(cell) × output_gate ────────┘
```

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class LSTMCell:
    def __init__(self, input_size, hidden_size):
        s = 0.3
        # 4 sets of weights: forget, input, candidate, output
        self.Wf = np.random.randn(input_size + hidden_size, hidden_size) * s
        self.Wi = np.random.randn(input_size + hidden_size, hidden_size) * s
        self.Wc = np.random.randn(input_size + hidden_size, hidden_size) * s
        self.Wo = np.random.randn(input_size + hidden_size, hidden_size) * s
        self.bf = np.zeros((1, hidden_size))
        self.bi = np.zeros((1, hidden_size))
        self.bc = np.zeros((1, hidden_size))
        self.bo = np.zeros((1, hidden_size))
    
    def forward(self, x, h_prev, c_prev):
        combined = np.hstack([x, h_prev])
        
        # Gates (sigmoid → values between 0 and 1)
        f = sigmoid(combined @ self.Wf + self.bf)   # forget gate
        i = sigmoid(combined @ self.Wi + self.bi)   # input gate
        o = sigmoid(combined @ self.Wo + self.bo)   # output gate
        
        # Candidate new memory
        c_candidate = tanh(combined @ self.Wc + self.bc)
        
        # Update cell state
        c_new = f * c_prev + i * c_candidate  # forget old + add new
        
        # Output
        h_new = o * tanh(c_new)
        
        return h_new, c_new, f, i, o

np.random.seed(42)
lstm = LSTMCell(input_size=3, hidden_size=4)

h = np.zeros((1, 4))  # hidden state
c = np.zeros((1, 4))  # cell state (long-term memory)

print("LSTM Processing: 'the cat sat'\n")

for word, x in words.items():
    h, c, f_gate, i_gate, o_gate = lstm.forward(x, h, c)
    print(f"  '{word}':")
    print(f"    Forget gate: {f_gate.flatten().round(2)}  (what to erase)")
    print(f"    Input gate:  {i_gate.flatten().round(2)}  (what to store)")
    print(f"    Output gate: {o_gate.flatten().round(2)}  (what to output)")
    print(f"    Cell state:  {c.flatten().round(3)}  (long-term memory)")
    print(f"    Hidden:      {h.flatten().round(3)}  (short-term output)")
    print()

In [ ]:
# Compare RNN vs LSTM memory retention
print("Memory Retention: RNN vs LSTM\n")

np.random.seed(42)

# RNN
h_rnn = np.zeros((1, hidden_size))
h_rnn = rnn_cell(strong_input, h_rnn, W_xh, W_hh, b_h)

# LSTM
lstm2 = LSTMCell(3, 4)
h_lstm = np.zeros((1, 4))
c_lstm = np.zeros((1, 4))
h_lstm, c_lstm, _, _, _ = lstm2.forward(strong_input, h_lstm, c_lstm)

rnn_init = np.linalg.norm(h_rnn)
lstm_init = np.linalg.norm(c_lstm)

print(f"{'Step':>5} {'RNN memory':>12} {'LSTM memory':>13}")
print(f"{'─'*5} {'─'*12} {'─'*13}")

for step in range(21):
    rnn_str = np.linalg.norm(h_rnn)
    lstm_str = np.linalg.norm(c_lstm)
    
    rnn_bar = "█" * int(rnn_str / rnn_init * 15)
    lstm_bar = "█" * int(lstm_str / lstm_init * 15)
    
    print(f"{step:>5} {rnn_str:>7.3f} {rnn_bar:<5} {lstm_str:>7.3f} {lstm_bar}")
    
    h_rnn = rnn_cell(zero_input, h_rnn, W_xh, W_hh, b_h)
    h_lstm, c_lstm, _, _, _ = lstm2.forward(zero_input, h_lstm, c_lstm)

print("\nLSTM retains information much longer than basic RNN!")
print("The cell state (conveyor belt) preserves info unless gates erase it.")

---
## GRU: A Simpler Alternative to LSTM

GRU (Gated Recurrent Unit) is a simplified LSTM:
- Combines forget + input gates into one **update gate**
- No separate cell state (just hidden state)
- Fewer parameters, often similar performance

```
LSTM: 3 gates + cell state  → more parameters, more expressive
GRU:  2 gates, no cell state → simpler, faster
```

In [ ]:
print("LSTM vs GRU Comparison:\n")
print(f"{'Property':<25} {'LSTM':<20} {'GRU':<20}")
print(f"{'─'*25} {'─'*20} {'─'*20}")
print(f"{'Gates':<25} {'3 (forget,input,out)':<20} {'2 (update,reset)':<20}")
print(f"{'Memory':<25} {'cell + hidden state':<20} {'hidden state only':<20}")
print(f"{'Parameters (h=256)':<25} {'~786K':<20} {'~590K':<20}")
print(f"{'Speed':<25} {'slower':<20} {'faster':<20}")
print(f"{'Long sequences':<25} {'slightly better':<20} {'good':<20}")
print(f"{'Popularity':<25} {'very common':<20} {'common':<20}")
print()
print("In practice: try GRU first (simpler), switch to LSTM if needed.")
print("But both are largely replaced by Transformers (Ch 12) now.")

---
## Practical Example: Predicting the Next Character

A simple sequence model that learns to predict the next character in a pattern.

In [ ]:
# Character-level sequence prediction
# Pattern: "abcabc..." — can the model learn to predict the next char?

text = "abcabcabcabcabcabcabcabc"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Text: '{text}'")
print(f"Vocabulary: {chars} ({vocab_size} chars)")
print(f"Encoding: {char_to_idx}\n")

# One-hot encode
def one_hot(idx, size):
    v = np.zeros((1, size))
    v[0, idx] = 1.0
    return v

# Simple RNN training
hidden_size = 8
np.random.seed(42)
Wxh = np.random.randn(vocab_size, hidden_size) * 0.3
Whh = np.random.randn(hidden_size, hidden_size) * 0.3
Why = np.random.randn(hidden_size, vocab_size) * 0.3
bh = np.zeros((1, hidden_size))
by = np.zeros((1, vocab_size))

lr = 0.1

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

print("Training RNN to predict next character...\n")

for epoch in range(501):
    total_loss = 0
    h = np.zeros((1, hidden_size))
    
    # Store for backprop
    hs = {-1: h.copy()}
    xs, ys_hat = {}, {}
    
    # Forward pass through sequence
    for t in range(len(text) - 1):
        xs[t] = one_hot(char_to_idx[text[t]], vocab_size)
        h = tanh(xs[t] @ Wxh + h @ Whh + bh)
        hs[t] = h.copy()
        y_hat = softmax(h @ Why + by)
        ys_hat[t] = y_hat
        target_idx = char_to_idx[text[t + 1]]
        total_loss -= np.log(y_hat[0, target_idx] + 1e-8)
    
    # Backward pass (truncated BPTT)
    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)
    dh_next = np.zeros_like(h)
    
    for t in reversed(range(len(text) - 1)):
        dy = ys_hat[t].copy()
        target_idx = char_to_idx[text[t + 1]]
        dy[0, target_idx] -= 1
        
        dWhy += hs[t].T @ dy
        dby += dy
        
        dh = dy @ Why.T + dh_next
        dh_raw = (1 - hs[t] ** 2) * dh
        
        dWxh += xs[t].T @ dh_raw
        dWhh += hs[t-1].T @ dh_raw
        dbh += dh_raw
        dh_next = dh_raw @ Whh.T
    
    # Clip gradients
    for d in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(d, -5, 5, out=d)
    
    Wxh -= lr * dWxh
    Whh -= lr * dWhh
    Why -= lr * dWhy
    bh -= lr * dbh
    by -= lr * dby
    
    if epoch % 100 == 0:
        avg_loss = total_loss / (len(text) - 1)
        
        # Generate predictions
        h_test = np.zeros((1, hidden_size))
        predictions = ""
        for t in range(len(text) - 1):
            x_test = one_hot(char_to_idx[text[t]], vocab_size)
            h_test = tanh(x_test @ Wxh + h_test @ Whh + bh)
            y_test = softmax(h_test @ Why + by)
            pred_char = idx_to_char[np.argmax(y_test)]
            predictions += pred_char
        
        correct = sum(p == text[i+1] for i, p in enumerate(predictions))
        print(f"  Epoch {epoch:>3} | Loss: {avg_loss:.3f} | Accuracy: {correct}/{len(predictions)}")
        print(f"           Input:      {text[:-1]}")
        print(f"           Predicted:  {predictions}")
        print(f"           Expected:   {text[1:]}")
        print()

---
## Limitations of Sequence Models

RNNs/LSTMs were state-of-the-art for language until 2017. But they have fundamental problems:

In [ ]:
print("Why Sequence Models Were Replaced:\n")

problems = [
    ("Sequential processing",
     "Must process word-by-word in order",
     "Cannot parallelize → slow on GPUs"),
    ("Long-range dependencies",
     "Even LSTM struggles with 1000+ tokens",
     "Information decays over distance"),
    ("Fixed bottleneck",
     "Entire sequence squeezed into one hidden vector",
     "A 256-dim vector can't capture a whole book"),
]

for name, what, why in problems:
    print(f"  Problem: {name}")
    print(f"    What:  {what}")
    print(f"    Why:   {why}")
    print()

print("The solution: ATTENTION (Ch 12)")
print("  → Process all words in parallel")
print("  → Any word can directly attend to any other word")
print("  → No information bottleneck")
print("\nBut understanding RNN/LSTM is important:")
print("  → They introduced the idea of processing sequences")
print("  → Gates (LSTM) inspired attention mechanisms")
print("  → Still used in some real-time / streaming applications")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **RNN** | Processes sequences step-by-step with hidden state as memory |
| **Vanishing memory** | Basic RNN forgets early inputs in long sequences |
| **LSTM** | 3 gates (forget, input, output) + cell state preserve long-term memory |
| **GRU** | Simplified LSTM with 2 gates, often similar performance |
| **Sequential bottleneck** | Must process in order → can't parallelize |
| **Replaced by** | Transformers (2017) solved all three limitations |

**Next chapter**: Tokenization — how text becomes numbers that models can process